# Part 2.2 - Relation discovery and relation classification
This notebook implements the reproducible B0/B1/B2 baseline protocol from `PART2_PLAN.md`. Splits are performed by debate, not by pair, and exact text overlap with the training set is removed from the test set.

In [ ]:
from pathlib import Path
import pandas as pd
from part2_relations import (make_discussion_split, apply_discussion_split,
                             majority_baseline, train_tfidf, evaluate_tfidf,
                             build_candidate_pairs, rank_candidates, ranking_metrics,
                             make_discovery_negatives, TfidfInteractionModel,
                             ContextTfidfInteractionModel, evaluate_discovery_binary)

DATA_DIR = Path('data')
claims_path = DATA_DIR / 'kialo_claims.csv'
pairs_path = DATA_DIR / 'kialo_pairs.csv'
if not claims_path.exists() or not pairs_path.exists():
    raise FileNotFoundError('Run part2_1_kialo_collection.ipynb and provide the Kialo CSV files first.')
claims = pd.read_csv(claims_path)
pairs = pd.read_csv(pairs_path)
print('claims:', len(claims), 'pairs:', len(pairs))

# One split is shared by classification and discovery.
split_ids = make_discussion_split(pairs['discussion_id'], train_fraction=.70, validation_fraction=.15, seed=42)

claims: 94707 pairs: 94199


## 1. Debate-level split and overlap control

In [ ]:
split = apply_discussion_split(pairs, split_ids)
print({name: len(frame) for name, frame in [('train', split.train), ('validation', split.validation), ('test', split.test)]})
print({name + ' debates': len(ids) for name, ids in [('train', split.train_ids), ('validation', split.validation_ids), ('test', split.test_ids)]})

{'train': 69333, 'validation': 10162, 'test': 14450}
{'train debates': 175, 'validation debates': 37, 'test debates': 38}


## 2. Relation classification: B0 and B1
The target is `support` versus `attack` for a linked child-parent pair. Report macro-F1 as the primary metric, class F1, accuracy, ROC-AUC/PR-AUC, and the confusion matrix.

In [ ]:
b0 = majority_baseline(split.train, split.test)
b1_model = train_tfidf(split.train)
b1 = evaluate_tfidf(b1_model, split.test)
display(pd.DataFrame([b0, b1]))

,model,accuracy,macro_f1,confusion_matrix,f1_attack,f1_support,roc_auc,pr_auc
0,B0 majority,0.504221,0.335204,"[[7286, 0], [7164, 0]]",0.670409,0.000000,NaN,NaN
1,B1 TF-IDF + logistic regression,0.623737,0.623688,"[[4589, 2697], [2740, 4424]]",0.627985,0.619391,0.66666,0.63568


## 3. Relation discovery
Discovery is evaluated in two ways: balanced linked/non-linked classification, and ranking all same-debate parent candidates with Hits@1, Hits@5, and MRR.

In [ ]:
train_claims = claims[claims['discussion_id'].astype(str).isin(split_ids['train'])].copy()
train_pairs = pairs[pairs['discussion_id'].astype(str).isin(split_ids['train'])].copy()
test_claims = claims[claims['discussion_id'].astype(str).isin(split_ids['test'])].copy()
test_pairs = pairs[pairs['discussion_id'].astype(str).isin(split_ids['test'])].copy()

# Build negatives independently inside each split, so cross-debate negatives
# cannot pull claims from another split.
train_discovery = make_discovery_negatives(train_claims, train_pairs, seed=42)
test_discovery = make_discovery_negatives(test_claims, test_pairs, seed=42)
discovery_split = apply_discussion_split(
    pd.concat([train_discovery, test_discovery], ignore_index=True), split_ids
)
discovery_train = discovery_split.train
discovery_test = discovery_split.test
discovery_model = TfidfInteractionModel().fit(discovery_train)
discovery_binary = evaluate_discovery_binary(discovery_model, discovery_test)
display(pd.DataFrame([discovery_binary]))
print('Train negative levels:', discovery_train['difficulty'].value_counts().to_dict())
print('Test negative levels:', discovery_test['difficulty'].value_counts().to_dict())

,model,accuracy,macro_f1,confusion_matrix,f1_linked,f1_not_linked,roc_auc,pr_auc
0,B1 interaction discovery,0.746809,0.690907,"[[9346, 5104], [9615, 34069]]",0.559456,0.822357,0.777961,0.525174


Train negative levels: {'same_debate': 69614, 'local': 69614, 'cross_debate': 69614, 'positive': 69333}
Test negative levels: {'local': 14586, 'same_debate': 14552, 'cross_debate': 14546, 'positive': 14450}


In [ ]:
def rank_and_summarize_by_debate(model, debate_ids):
    per_debate = []
    for debate_id in sorted(map(str, debate_ids)):
        candidates = build_candidate_pairs(
            claims, pairs, discussion_ids={debate_id}
        )
        if candidates.empty:
            continue
        ranked_debate = rank_candidates(model, candidates)
        metrics = ranking_metrics(ranked_debate)
        metrics['discussion_id'] = debate_id
        per_debate.append(metrics)

    per_debate_frame = pd.DataFrame(per_debate)
    if per_debate_frame.empty:
        return per_debate_frame, {}

    # Each debate contributes in proportion to the number of child groups scored.
    weights = per_debate_frame['groups']
    weighted = {
        metric: (per_debate_frame[metric] * weights).sum() / weights.sum()
        for metric in ('mean_candidates', 'hits_at_1', 'hits_at_5', 'mrr')
    }
    weighted['groups'] = int(weights.sum())
    weighted['debates'] = int(len(per_debate_frame))
    return per_debate_frame, weighted

if split.test_ids:
    ranked_by_debate, ranking_summary = rank_and_summarize_by_debate(
        discovery_model, split.test_ids
    )
    print('Relation discovery ranking, per debate:')
    display(ranked_by_debate)
    print('Mean weighted by groups:')
    display(pd.DataFrame([ranking_summary]))

    context_model = ContextTfidfInteractionModel().fit(discovery_train)
    context_by_debate, context_summary = rank_and_summarize_by_debate(
        context_model, split.test_ids
    )
    print('GraphNLI-inspired context ranking, per debate:')
    display(context_by_debate)
    print('Context mean weighted by groups:')
    display(pd.DataFrame([context_summary]))
else:
    print('No ranking candidates available for the test debates.')

Relation discovery ranking, per debate:


,groups,mean_candidates,hits_at_1,hits_at_5,mrr,discussion_id
0,50,50.0,0.100000,0.320000,0.233042,11280
1,474,474.0,0.122363,0.265823,0.195374,1324
2,204,204.0,0.122549,0.343137,0.241365,13439
3,246,246.0,0.105691,0.256098,0.190720,13864
4,974,974.0,0.129363,0.264887,0.199559,14247
5,195,233.0,0.128205,0.328205,0.224894,14768
6,140,140.0,0.142857,0.342857,0.250049,15546
7,130,130.0,0.161538,0.376923,0.268160,15599
8,478,481.0,0.092050,0.221757,0.159496,1592
9,503,503.0,0.131213,0.296223,0.214865,1634


Mean weighted by groups:


,mean_candidates,hits_at_1,hits_at_5,mrr,groups,debates
0,745.452151,0.126684,0.289007,0.209908,14619,38


GraphNLI-inspired context ranking, per debate:


,groups,mean_candidates,hits_at_1,hits_at_5,mrr,discussion_id
0,50,50.0,0.020000,0.100000,0.096875,11280
1,474,474.0,0.120253,0.244726,0.187106,1324
2,204,204.0,0.127451,0.357843,0.241593,13439
3,246,246.0,0.101626,0.252033,0.180292,13864
4,974,974.0,0.129363,0.226899,0.183729,14247
5,195,233.0,0.128205,0.328205,0.218740,14768
6,140,140.0,0.121429,0.364286,0.228478,15546
7,130,130.0,0.146154,0.346154,0.250437,15599
8,478,481.0,0.094142,0.209205,0.156513,1592
9,503,503.0,0.129225,0.246521,0.189976,1634


Context mean weighted by groups:


,mean_candidates,hits_at_1,hits_at_5,mrr,groups,debates
0,745.452151,0.125385,0.262402,0.196107,14619,38


## 4. B2 frozen Sentence-BERT
B2 is optional and requires `sentence-transformers`. It uses frozen embeddings and the feature vector `(u, v, |u-v|, u*v)` followed by logistic regression. Keep the same split and test set as B1.

In [ ]:
try:
    from part2_relations import SentenceEmbeddingPairModel, evaluate_sentence_embeddings
    b2_model = SentenceEmbeddingPairModel()
    b2_model.fit(split.train)
    display(pd.DataFrame([evaluate_sentence_embeddings(b2_model, split.test)]))
except ImportError as error:
    print('B2 skipped:', error)

/private/tmp/claude-501/-Users-mac-Desktop-Master-Lyon-M2-LYON1-Dynamique-des-connaisance-TP-projetarg/32b53c19-1fd1-4563-8f44-0a8fa6a7fd21/scratchpad/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,model,accuracy,macro_f1,confusion_matrix,f1_attack,f1_support,roc_auc,pr_auc
0,B2 frozen Sentence-BERT + logistic regression,0.649481,0.649479,"[[4674, 2612], [2453, 4711]]",0.648581,0.650376,0.709186,0.688101









## 5. Critical evaluation

Compare models with identical debate-level splits and seeds. Report performance by depth and topic, 20 representative errors, the number of candidates per child, and the random baseline for Hits@1. Compare cautiously with GraphNLI, Gorur et al., Faugier et al., and Ruiz-Dolz et al.; their snapshots, domains, labels, and split protocols differ.